# CS342 - Lab Assignment 1 (Single Image Version)
All 3 tasks on **one image** — segmentation, metrics, edge detection.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color, filters, segmentation, morphology, feature
from skimage.filters import sobel, roberts, prewitt, scharr, laplace, threshold_otsu
from skimage.segmentation import slic, felzenszwalb, watershed, mark_boundaries
from skimage.color import rgb2gray, label2rgb
from skimage.feature import canny
from skimage.metrics import variation_of_information, adapted_rand_error
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

## Load Image
Using `skimage.data.chelsea()` (a cat photo, 300x451 RGB). No download needed.

In [ ]:
img = data.chelsea()  # Change to data.astronaut(), data.coffee(), etc.
gray = rgb2gray(img)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img); axes[0].set_title('Original RGB'); axes[0].axis('off')
axes[1].imshow(gray, cmap='gray'); axes[1].set_title('Grayscale'); axes[1].axis('off')
axes[2].hist(gray.ravel(), bins=64, color='steelblue', edgecolor='black')
axes[2].set_title('Intensity Histogram'); axes[2].set_xlabel('Pixel Value')
print(f'Shape: {img.shape} | Dtype: {img.dtype} | Range: [{img.min()}, {img.max()}]')
plt.tight_layout(); plt.show()

---
## Task 2: Segmentation Algorithms

In [ ]:
# Apply 4 segmentation methods
seg_slic = slic(img, n_segments=100, compactness=10, start_label=1)
seg_felz = felzenszwalb(img, scale=100, sigma=0.5, min_size=50)

gradient = sobel(gray)
markers = np.zeros_like(gray, dtype=int)
markers[gray < 0.3] = 1
markers[gray > 0.7] = 2
seg_water = watershed(gradient, markers)

thresh = threshold_otsu(gray)
seg_otsu = (gray > thresh).astype(int) + 1

results = [('SLIC', seg_slic), ('Felzenszwalb', seg_felz),
           ('Watershed', seg_water), ('Otsu', seg_otsu)]

for name, seg in results:
    print(f'{name}: {len(np.unique(seg))} segments')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, (name, seg) in enumerate(results):
    axes[0, i].imshow(mark_boundaries(img, seg, color=(1, 0, 0)))
    axes[0, i].set_title(f'{name} — Boundaries'); axes[0, i].axis('off')
    axes[1, i].imshow(label2rgb(seg, img, kind='avg', bg_label=0))
    axes[1, i].set_title(f'{name} — Avg Color'); axes[1, i].axis('off')
plt.suptitle('Segmentation Results', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

### Distance Metrics Between Segmentations

In [ ]:
names = ['SLIC', 'Felzenszwalb', 'Watershed', 'Otsu']
segs = [seg_slic, seg_felz, seg_water, seg_otsu]
n = len(names)

vi_mat = np.zeros((n, n))
rand_mat = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        if i != j:
            s, m = variation_of_information(segs[i], segs[j])
            vi_mat[i, j] = s + m
            are, _, _ = adapted_rand_error(segs[i], segs[j])
            rand_mat[i, j] = are

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, mat, title in zip(axes, [vi_mat, rand_mat],
    ['Variation of Information (lower=similar)', 'Adapted Rand Error (lower=similar)']):
    im = ax.imshow(mat, cmap='YlOrRd')
    ax.set_xticks(range(n)); ax.set_xticklabels(names, rotation=45)
    ax.set_yticks(range(n)); ax.set_yticklabels(names)
    ax.set_title(title, fontsize=12)
    for r in range(n):
        for c in range(n):
            ax.text(c, r, f'{mat[r,c]:.2f}', ha='center', va='center', fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

In [ ]:
# Dice coefficient (binary comparison)
def dice(s1, s2):
    b1 = (s1 > np.median(s1)).astype(bool)
    b2 = (s2 > np.median(s2)).astype(bool)
    return 2 * np.logical_and(b1, b2).sum() / (b1.sum() + b2.sum())

print('Dice Coefficient (higher = more similar):')
print(f'{"":>14s}', ''.join(f'{n:>14s}' for n in names))
for i in range(n):
    row = ''.join(f'{dice(segs[i], segs[j]):>14.4f}' for j in range(n))
    print(f'{names[i]:>14s}{row}')

---
## Task 3: Edge Detection

In [ ]:
edges = {
    'Sobel': sobel(gray),
    'Prewitt': prewitt(gray),
    'Roberts': roberts(gray),
    'Scharr': scharr(gray),
    'Laplacian': np.abs(laplace(gray)),
    'Canny σ=1': canny(gray, sigma=1).astype(float),
    'Canny σ=2': canny(gray, sigma=2).astype(float),
    'Canny σ=3': canny(gray, sigma=3).astype(float),
}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, (name, e) in zip(axes.ravel(), edges.items()):
    ax.imshow(e, cmap='gray'); ax.set_title(name, fontsize=12); ax.axis('off')
plt.suptitle('Edge Detection Comparison', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Edge statistics
th = 0.1
print(f'{"Detector":<15s} {"Edge Pixels":>12s} {"Density %":>10s}')
print('-' * 40)
for name, e in edges.items():
    binary = e > 0.5 if 'Canny' in name else e > th
    count = binary.sum()
    density = 100 * count / binary.size
    print(f'{name:<15s} {count:>12d} {density:>9.2f}%')

In [ ]:
# Noise robustness test
np.random.seed(42)
noisy = np.clip(gray + 0.1 * np.random.randn(*gray.shape), 0, 1)

noisy_edges = {
    'Sobel': sobel(noisy),
    'Roberts': roberts(noisy),
    'Canny σ=1': canny(noisy, sigma=1).astype(float),
    'Canny σ=3': canny(noisy, sigma=3).astype(float),
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(noisy, cmap='gray'); axes[0].set_title('Noisy Input'); axes[0].axis('off')
for ax, (name, e) in zip(axes[1:], noisy_edges.items()):
    ax.imshow(e, cmap='gray'); ax.set_title(name); ax.axis('off')
plt.suptitle('Edge Detection on Noisy Image', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# False edge rate
print('False Edge Rate (noisy vs clean):')
print('-' * 55)
for name in noisy_edges:
    clean = edges[name] > 0.5 if 'Canny' in name else edges[name] > th
    noisy_b = noisy_edges[name] > 0.5 if 'Canny' in name else noisy_edges[name] > th
    clean_d = morphology.binary_dilation(clean, morphology.disk(2))
    false_e = np.logical_and(noisy_b, ~clean_d).sum()
    total = noisy_b.sum()
    rate = 100 * false_e / total if total > 0 else 0
    print(f'{name:<15s} | False: {false_e:>5d} | Total: {total:>5d} | Rate: {rate:.1f}%')

---
## Conclusions
- **Segmentation**: SLIC/Felzenszwalb give fine superpixels; Otsu is simple binary; Watershed needs good markers.
- **Edge Detection**: Canny (σ=2) gives best localization with fewest false edges. Roberts is sharp but noise-sensitive. Sobel/Prewitt are balanced.
- **Noise**: Higher Canny σ suppresses noise but loses fine detail — σ=2 is a good tradeoff.